# #2 Lineage Statistics

## Purpose

Calculate topology and recording statistics for each embryo.

## Setup

In [ ]:
import treedata as td
import pycea as py
import pandas as pd
import numpy as np

from devmap.config import set_theme, get_paths, embryos
from devmap.utils import load_data

set_theme()
base_path, plots_path, results_path = get_paths("trees")
data_path = base_path / "data"

## Load data

In [ ]:
tdata = load_data("topology")
characters = pd.read_csv(data_path / "characters.csv", index_col=0)

/tmp/ipykernel_1156353/3984897116.py:2: DtypeWarning: Columns (61,62,63,100,101,102,103,104,105) have mixed types. Specify dtype option on import or set low_memory=False.
  characters = pd.read_csv(data_path / "characters.csv", index_col=0)


## Total edges

In [20]:
total_edges = 0
for clone, tree in tdata.obst.items():
    total_edges += tree.number_of_edges()
print(f"Total edges: {total_edges}")

Total edges: 2177176


## Calculate embryo statistics

depth

In [24]:
py.pp.add_depth(tdata)

Uniquely marked leaves

In [25]:
unique_leaves = {}
for embryo, df in tdata.obs.query('clone.notnull()').groupby("embryo"):
    embryo_characters = characters.loc[df.index].copy()
    embryo_characters = embryo_characters.replace("-",pd.NA)
    n_unique = embryo_characters.drop_duplicates().shape[0]
    unique_leaves[embryo] = n_unique/embryo_characters.shape[0]

/tmp/ipykernel_1156353/638010538.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  embryo_characters = embryo_characters.replace("-",pd.NA)


Branches with an edit

In [29]:
def fraction_branches_marked(G):
    leaves = sum(1 for n in G.nodes if G.out_degree(n) == 0)
    edges = G.number_of_edges()
    return edges / (2 * leaves - 2)

branches_marked = {}
for clone, tree in tdata.obst.items():
    branches_marked[clone] = fraction_branches_marked(tree)
branches_marked = pd.DataFrame.from_dict(branches_marked, orient="index", columns=["branches_marked"])
tdata.obs["n"] = 1
branches_marked["clone_size"] = tdata.obs.groupby("clone")["n"].sum()
branches_marked["embryo"] = branches_marked.index.str.split("-C").str[0]
branches_marked = branches_marked.groupby("embryo").apply(lambda x: np.average(x["branches_marked"], weights=x["clone_size"]))

/tmp/ipykernel_1156353/1430023880.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  branches_marked = branches_marked.groupby("embryo").apply(lambda x: np.average(x["branches_marked"], weights=x["clone_size"]))


Total cells

In [30]:
total_cells =  {
    "E7.5-R1": 3825,
    "E7.5-R2": 6750,
    "E7.5-R3": 7687,
    "E8.0-R1": 14375,
    "E8.0-R2": 19375,
    "E8.0-R3": 12500,
    "E8.5-R1": 55000,
    "E8.5-R2": 91250,
    "E8.5-R3": 46250,
    "E9.0-R1": 187500,
    "E9.0-R2": 256250,
    "E9.0-R3": 134375,
    "E9.5-R1": 759360,
    "E9.5-R2": 696864,
    "E9.5-R3": 510000,
    "E10.0-R1": 971875,
}

All stats

In [ ]:
embryo_stats = tdata.obs.groupby("embryo").agg(
    n_cells = ("n", "sum"),
    n_tracing = ("clone", lambda x: x.notna().sum()),
    n_captures = ("capture", "nunique"),
    mean_umis = ("total_counts", "mean"),
    n_clones = ("clone","nunique"),
    chimerism_rate = ("type", lambda x: (x.dropna() == "donor").mean()),
    detection_rate = ("detection_rate", "mean"),
    edit_frac = ("edit_frac", "mean"),
    avg_tree_depth = ("depth", "mean"),
)
embryo_stats = embryo_stats.loc[embryos,:].copy()
embryo_stats["branches_marked"] = branches_marked
embryo_stats["unique_leaves"] = embryo_stats.index.map(unique_leaves)
embryo_stats["cell_total_estimate"] = embryo_stats.index.map(total_cells)
embryo_stats["recovery_estimate"] = embryo_stats["n_cells"] / embryo_stats["cell_total_estimate"]
embryo_stats.round(2).to_csv(results_path / "lineage_stats.csv", index=True)